# DP and EMT three-phase SSN grid validation

Run the paired C++ examples that use the three-phase SSN transformer, PiLine, and impedance load. The checks validate finite results, expected voltage/current levels, and agreement between DP steady-state phasors and EMT waveforms.

In [ ]:
import os
import subprocess
from pathlib import Path

import numpy as np
from villas.dataprocessing.readtools import read_timeseries_csv

In [ ]:
ROOT = Path.cwd()
EXAMPLES = Path(
    os.environ.get("DPSIM_CXX_EXAMPLES_DIR", ROOT / "build/dpsim/examples/cxx")
).resolve()

simulations = {
    "DP": "DP_Ph3_SSN_Grid_Validation",
    "EMT": "EMT_Ph3_SSN_Grid_Validation",
}
for name in simulations.values():
    executable = EXAMPLES / name
    assert executable.is_file(), f"Missing C++ example: {executable}"
    subprocess.run([str(executable)], cwd=ROOT, check=True)

results = {
    domain: read_timeseries_csv(
        ROOT / "logs" / name / f"{name}.csv", print_status=False
    )
    for domain, name in simulations.items()
}

In [ ]:
def signal_values(domain, signal):
    matches = [
        np.asarray(series.values)
        for key, series in results[domain].items()
        if key.startswith(signal)
    ]
    assert matches, f"Missing {signal} signals in {domain} output"
    return np.column_stack(matches)


for domain in simulations:
    for signal in ("source_bus_v", "transformer_bus_v", "load_bus_v", "load_i"):
        values = signal_values(domain, signal)
        assert values.ndim == 2 and values.shape[0] > 10
        assert np.isfinite(values).all(), f"Non-finite {signal} in {domain}"

# DP values are RMS phasors. Compare the last-cycle EMT RMS magnitude.
for signal in ("source_bus_v", "transformer_bus_v", "load_bus_v", "load_i"):
    dp = signal_values("DP", signal)
    emt = signal_values("EMT", signal)
    dp_mag = np.mean(np.abs(dp[-100:]), axis=0)
    emt_rms = np.sqrt(np.mean(emt[-1000:] ** 2, axis=0))
    np.testing.assert_allclose(dp_mag, emt_rms, rtol=0.08, atol=2.0)

transformer_voltage = signal_values("DP", "transformer_bus_v")
load_voltage = signal_values("DP", "load_bus_v")
load_current = signal_values("DP", "load_i")
for voltage in (transformer_voltage, load_voltage):
    assert np.all(
        (np.mean(np.abs(voltage[-100:]), axis=0) > 100)
        & (np.mean(np.abs(voltage[-100:]), axis=0) < 160)
    )

assert np.all(
    (np.mean(np.abs(load_current[-100:]), axis=0) > 10)
    & (np.mean(np.abs(load_current[-100:]), axis=0) < 200)
)

print("DP and EMT three-phase SSN grid checks passed")